### UNSLOTH PHI-3.5 MODEL

In [ ]:
#from google.colab import drive -- Uncomment for Colab
import os

#drive.mount('/content/drive') -- Uncomment for Colab
BASE_DIR = "./phi-3.5-2"

CHECKPOINT_DIR = os.path.join(BASE_DIR, "checkpoints")
FINAL_MODEL_DIR = os.path.join(BASE_DIR, "inference_model")
MODEL_MERGED_DIR = os.path.join(BASE_DIR, "merged_model")
GGUF_DIR = os.path.join(BASE_DIR, "gguf_export")

HUGGINGFACE_MODEL_DIR  = "lauraloretta/phi3-merged-mini-2.0"
HUGGINGFACE_HUB_TOKEN = "YOUR-TOKEN"

print("BASE_DIR:", BASE_DIR)
print("CHECKPOINT_DIR:", CHECKPOINT_DIR)
print("FINAL_MODEL_DIR:", FINAL_MODEL_DIR)
print("GGUF_DIR:", GGUF_DIR)

BASE_DIR: ./phi-3.5-2
CHECKPOINT_DIR: ./phi-3.5-2/checkpoints
FINAL_MODEL_DIR: ./phi-3.5-2/inference_model
GGUF_DIR: ./phi-3.5-2/gguf_export


In [2]:
%%capture
!pip install unsloth
# Also get the latest nightly Unsloth!
!pip uninstall unsloth -y && pip install --upgrade --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git@nightly git+https://github.com/unslothai/unsloth-zoo.git

In [ ]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Phi-3.5-mini-instruct", 
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.11.6: Fast Llama patching. Transformers: 4.57.2.
   \\   /|    NVIDIA GeForce RTX 4070 Laptop GPU. Num GPUs = 1. Max memory: 7.996 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.5.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


We now add LoRA adapters so we only need to update 1 to 10% of all parameters!

In [3]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

Unsloth 2025.11.6 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


<a name="Data"></a>
### Data Prep

We use our `get_chat_template` function to get the correct chat template.

In [4]:
from unsloth.chat_templates import get_chat_template, standardize_sharegpt
from datasets import load_dataset

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "phi-3",
)

def formatting_prompts_func(examples):
    convos = examples["conversations"]
    texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False) for convo in convos]
    return { "text" : texts, }



In [5]:
from datasets import load_dataset

dataset = load_dataset("mlabonne/FineTome-100k", split="train")

# Standardize to ShareGPT format (required by Unsloth)
dataset = standardize_sharegpt(dataset)

print("Dataset size BEFORE subset:", len(dataset))

# Take subset of 10k for faster training (your choice)
SUBSET_SIZE = 10000
dataset = dataset.shuffle(seed=42).select(range(SUBSET_SIZE))

print("Dataset size AFTER subset:", len(dataset))

# Convert conversations → formatted text prompts
dataset = dataset.map(
    formatting_prompts_func,
    batched=True,
    remove_columns=[col for col in dataset.column_names if col != "text"],
)

# SHOW one sample to verify formatting
print(dataset[0]["text"])


Dataset size BEFORE subset: 100000
Dataset size AFTER subset: 10000
<|user|>
Give three types of computer graphics.<|end|>
<|assistant|>
1. Raster Graphics: These are also called bitmap graphics and are composed of pixels arranged in a grid. Each pixel can have a different color and shade. Raster graphics excel at representing photographic images and digital painting.

2. Vector Graphics: These graphics are constructed using mathematical formulas representing geometric shapes like lines, curves, and polygons. They are resolution-independent, meaning they can be scaled up or down in size without losing quality. Vector graphics are commonly used for logos, icons, typography and illustrations.

3. 3D Graphics: These graphics are used to create three-dimensional digital representations of objects. 3D graphics use techniques like modeling, rendering, and shading to simulate depth and surface properties. These graphics are used in animation, video games, architecture, engineering, and virtua

We look at how the text is structured for item 5:

In [6]:
dataset[5]["text"]

"<|user|>\nHow can I modify a C++ program to print out a Fibonacci sequence up to the nth number?<|end|>\n<|assistant|>\nYou can modify the given C++ program to print out a Fibonacci sequence up to the nth number. First, you need to declare and initialize a variable 'n' to specify the number of terms you want in the sequence. In the given code, 'n' is assigned a value of 10. \n\nNext, you declare three variables: 'first', 'second', and 'next'. 'first' and 'second' are initialized to 0 and 1 respectively, as these are the first two terms of the Fibonacci sequence.\n\nThen, you can use a for loop to iterate 'n' times and calculate the Fibonacci sequence. Inside the loop, you check if the current index 'i' is less than or equal to 1. If it is, 'next' is assigned the value of 'i'. This is because the first two terms of the Fibonacci sequence are 0 and 1. \n\nIf 'i' is greater than 1, 'next' is calculated by adding 'first' and 'second'. 'first' is then updated to the previous value of 'seco

<a name="Train"></a>
### Train the model

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments, DataCollatorForSeq2Seq
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    data_collator = DataCollatorForSeq2Seq(tokenizer = tokenizer),
    dataset_num_proc = 2,
    packing = True, # Can make training 5x faster for short sequences.
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 1, # Set this for 1 full training run.
        # max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 50,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        # output_dir = "outputs",
        output_dir=CHECKPOINT_DIR,
        save_strategy="steps",
        save_steps=200,
        save_total_limit=3,
        report_to = "none", # Use this for WandB etc
    ),
)

We also use Unsloth's `train_on_completions` method to only train on the assistant outputs and ignore the loss on the user's inputs.

In [8]:
from unsloth.chat_templates import train_on_responses_only

trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|user|>",
    response_part     = "<|assistant|>",
)


We verify masking is actually done:

In [9]:
tokenizer.decode(trainer.train_dataset[5]["input_ids"])

"<|user|> How can I modify a C++ program to print out a Fibonacci sequence up to the nth number?<|end|><|assistant|> You can modify the given C++ program to print out a Fibonacci sequence up to the nth number. First, you need to declare and initialize a variable 'n' to specify the number of terms you want in the sequence. In the given code, 'n' is assigned a value of 10. \n\nNext, you declare three variables: 'first', 'second', and 'next'. 'first' and 'second' are initialized to 0 and 1 respectively, as these are the first two terms of the Fibonacci sequence.\n\nThen, you can use a for loop to iterate 'n' times and calculate the Fibonacci sequence. Inside the loop, you check if the current index 'i' is less than or equal to 1. If it is, 'next' is assigned the value of 'i'. This is because the first two terms of the Fibonacci sequence are 0 and 1. \n\nIf 'i' is greater than 1, 'next' is calculated by adding 'first' and 'second'. 'first' is then updated to the previous value of 'second',

In [10]:
space = tokenizer(" ", add_special_tokens = False).input_ids[0]
tokenizer.decode([space if x == -100 else x for x in trainer.train_dataset[5]["labels"]])

"                                                      You can modify the given C++ program to print out a Fibonacci sequence up to the nth number. First, you need to declare and initialize a variable 'n' to specify the number of terms you want in the sequence. In the given code, 'n' is assigned a value of 10. \n\nNext, you declare three variables: 'first', 'second', and 'next'. 'first' and 'second' are initialized to 0 and 1 respectively, as these are the first two terms of the Fibonacci sequence.\n\nThen, you can use a for loop to iterate 'n' times and calculate the Fibonacci sequence. Inside the loop, you check if the current index 'i' is less than or equal to 1. If it is, 'next' is assigned the value of 'i'. This is because the first two terms of the Fibonacci sequence are 0 and 1. \n\nIf 'i' is greater than 1, 'next' is calculated by adding 'first' and 'second'. 'first' is then updated to the previous value of 'second', and 'second' is updated to the current value of 'next'. This 

We can see the System and Instruction prompts are successfully masked!

In [11]:
#@title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = NVIDIA GeForce RTX 4070 Laptop GPU. Max memory = 7.996 GB.
2.887 GB of memory reserved.


In [ ]:
from transformers.trainer_utils import get_last_checkpoint
import os

output_dir = CHECKPOINT_DIR

last_ckpt = None
if os.path.isdir(output_dir):
    last_ckpt = get_last_checkpoint(output_dir)

if last_ckpt is not None:
    print(f"Resuming from checkpoint: {last_ckpt}")
    trainer_stats = trainer.train(resume_from_checkpoint=last_ckpt)
else:
    print("No checkpoint found, starting from scratch.")
    trainer_stats = trainer.train()

The model is already on multiple devices. Skipping the move to device specified in `args`.


Resuming from checkpoint: ./phi-3.5-2/checkpoints/checkpoint-61


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 10,000 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 29,884,416 of 3,850,963,968 (0.78% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss


In [13]:
#@title Show final memory and time stats
import torch
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory         /max_memory*100, 3)
lora_percentage = round(used_memory_for_lora/max_memory*100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training.")
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

10.4675 seconds used for training.
0.17 minutes used for training.
Peak reserved memory = 2.887 GB.
Peak reserved memory for training = 0.0 GB.
Peak reserved memory % of max memory = 36.106 %.
Peak reserved memory for training % of max memory = 0.0 %.


<a name="Inference"></a>
### Inference
Let's run the model! 

In [14]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "phi-3",
)
FastLanguageModel.for_inference(model) # Enable native 2x faster inference

messages = [
    {"role": "user", "content": "Continue the fibonnaci sequence: 1, 1, 2, 3, 5, 8,"},
]
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True, # Must add for generation
    return_tensors = "pt",
).to(model.device)

outputs = model.generate(input_ids = inputs, max_new_tokens = 64, use_cache = True,
                         temperature = 1.5, min_p = 0.1)
tokenizer.batch_decode(outputs)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


['<|user|> Continue the fibonnaci sequence: 1, 1, 2, 3, 5, 8,<|end|><|assistant|> The Fibonacci sequence is a series of numbers in which each number is the sum of the two preceding ones. It usually starts with 0 and 1. In this case, it starts with 1 and 1.\n\nSo, the next number in the sequence would be the sum of the last']

 You can also use a `TextStreamer` for continuous inference - so you can see the generation token by token, instead of waiting the whole time!

In [15]:
FastLanguageModel.for_inference(model) # Enable native 2x faster inference

messages = [
    {"role": "user", "content": "Continue the fibonnaci sequence: 1, 1, 2, 3, 5, 8,"},
]
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True, # Must add for generation
    return_tensors = "pt",
).to(model.device)

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt = True)
_ = model.generate(input_ids = inputs, streamer = text_streamer, max_new_tokens = 128,
                   use_cache = True, temperature = 0.7, min_p = 0.1)

The Fibonacci sequence is a series of numbers in which each number is the sum of the two preceding ones. It usually starts with 0 and 1. In this case, it starts with 1 and 1.

So, the next number in the sequence would be the sum of the last two numbers, which are 5 and 8.

5 + 8 = 13

So, the next number in the sequence is 13.

The sequence would continue as follows: 1, 1, 2, 3, 5, 8, 


<a name="Save"></a>
### Saving, loading finetuned models


In [16]:
model.save_pretrained_merged(
    MODEL_MERGED_DIR,
    tokenizer,
    save_method = "merged_16bit"   # Or "merged_8bit" if you want smaller size
)

Found HuggingFace hub cache directory: /home/lpa/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model-00001-of-00002.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Merging weights into 16bit: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████| 2/2 [00:37<00:00, 18.61s/it]


Unsloth: Merge process complete. Saved to `/home/lpa/master/scalable_ML/phi-3.5-2/merged_model`


In [ ]:
from huggingface_hub import HfApi
import os
print(os.listdir(MODEL_MERGED_DIR))

api = HfApi()

api.upload_folder(
    folder_path=MODEL_MERGED_DIR,
    repo_id=HUGGINGFACE_MODEL_DIR,
    token=HUGGINGFACE_HUB_TOKEN,
    repo_type="model",
    commit_message="Upload merged Phi-3.5 model",
    ignore_patterns=[".cache", "*.lock"]
)



['config.json', 'tokenizer.model', '.cache', 'model.safetensors.index.json', 'tokenizer_config.json', 'model-00002-of-00002.safetensors', 'chat_template.jinja', 'model-00001-of-00002.safetensors', 'tokenizer.json', 'special_tokens_map.json', 'added_tokens.json']


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/lauraloretta/phi3-merged-mini-2.0/commit/88fa45c866ea51ae5cc83ab50944f86f1df76b9c', commit_message='Upload merged Phi-3.5 model', commit_description='', oid='88fa45c866ea51ae5cc83ab50944f86f1df76b9c', pr_url=None, repo_url=RepoUrl('https://huggingface.co/lauraloretta/phi3-merged-mini-2.0', endpoint='https://huggingface.co', repo_type='model', repo_id='lauraloretta/phi3-merged-mini-2.0'), pr_revision=None, pr_num=None)